In [ ]:

import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split


# Set the path to the directory containing the face images
faces_dir = "D:/R2021 DL LAB/Faces/Faces"


# Load the face images and labels
x_data = []
y_data = []


# Iterate over the face image directory
for filename in os.listdir(faces_dir):

    if filename.lower().endswith(".jpg"):

        img_path = os.path.join(faces_dir, filename)

        # Resize images to 64x64 pixels
        img = load_img(img_path, target_size=(64, 64))

        # Convert image to NumPy array
        img_array = img_to_array(img)

        x_data.append(img_array)

        # Extract label from filename
        # Example: person_1.jpg -> person_1
        label = filename.split(".")[0]

        y_data.append(label)


# Convert the data into NumPy arrays
x_data = np.array(x_data)
y_data = np.array(y_data)


# Normalize pixel values from 0-255 to 0-1
x_data = x_data.astype("float32") / 255.0


# Preprocess labels to extract numeric part
# Example: person_1 -> 1
y_data_numeric = np.array([
    int(label.split("_")[1])
    for label in y_data
])


# Find the number of classes
num_classes = len(np.unique(y_data_numeric))


# Convert labels to one-hot encoded vectors
y_data_encoded = tf.keras.utils.to_categorical(
    y_data_numeric,
    num_classes=num_classes
)


# Split the data into training and validation sets
x_train, x_val, y_train, y_val = train_test_split(
    x_data,
    y_data_encoded,
    test_size=0.2,
    random_state=42
)


# Define the CNN architecture for face recognition
model = tf.keras.models.Sequential([
    
    tf.keras.layers.Input(shape=(64, 64, 3)),

    tf.keras.layers.Conv2D(
        32,
        (3, 3),
        activation="relu"
    ),

    tf.keras.layers.MaxPooling2D((2, 2)),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(
        128,
        activation="relu"
    ),

    tf.keras.layers.Dense(
        num_classes,
        activation="softmax"
    )
])


# Compile the model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


# Display model architecture
model.summary()


# Train the CNN model
model.fit(
    x_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(x_val, y_val)
)


# Save the trained model
model.save("face_recognition_model.keras")

print("Model trained and saved successfully!")